In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
files.upload()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d shayanfazeli/heartbeat -p ./data --unzip

In [ ]:
!ls data

In [ ]:
# =============================================================================
#  Smart Hospital Edge AI — ECG Anomaly Detection
#  1D CNN Training Pipeline  |  PyTorch  |  Colab-ready  |  INT8-prep
#
#  Architecture : 3-block Conv1D → FC → Binary logit
#  Dataset      : MIT-BIH Arrhythmia (Kaggle pre-segmented CSVs)
#  Task         : Normal (0) vs Abnormal (1)
#  Window       : 187 samples / beat  (Pan-Tompkins segmentation)
#  Quantization : BatchNorm fused, INT8-export-ready via torch.ao.quantization
# =============================================================================
#
#  ── COLAB QUICK START ────────────────────────────────────────────────────────
#
#  Step 1 — Install dependencies (run once):
#    !pip install -q kaggle
#
#  Step 2 — Upload your kaggle.json, then download dataset:
#    from google.colab import files
#    files.upload()                          # upload kaggle.json
#    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
#    !kaggle datasets download -d shayanfazeli/heartbeat -p ./data --unzip
#
#  Step 3 — Run this script:
#    !python ecg_training_pipeline.py
#
#  Output files:
#    ecg_cnn_best.pth          ← best checkpoint (val loss)
#    ecg_cnn_final.pth         ← final epoch weights
#    training_curves.png       ← loss + accuracy plot
# =============================================================================

import os
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                        # headless-safe (Colab overrides this)
import matplotlib.pyplot as plt

from collections import Counter
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm import tqdm

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =============================================================================
#  SECTION 0 — Configuration (single place to tune everything)
# =============================================================================
CFG = {
    # Data
    "data_dir"       : "./data",
    "window_size"    : 187,           # samples per beat

    # Training
    "val_split"      : 0.10,          # fraction of train set used for validation
    "batch_size"     : 128,
    "epochs"         : 30,
    "lr"             : 1e-3,
    "weight_decay"   : 1e-4,
    "dropout"        : 0.4,

    # Hardware
    "num_workers"    : 2,
    "device"         : "cuda" if torch.cuda.is_available() else "cpu",

    # Outputs
    "best_ckpt"      : "ecg_cnn_best.pth",
    "final_ckpt"     : "ecg_cnn_final.pth",
    "plot_path"      : "training_curves.png",
}

print(f"[Config] Device : {CFG['device'].upper()}")
print(f"[Config] PyTorch: {torch.__version__}")


# =============================================================================
#  SECTION 1 — Dataset Loading & Binary Remapping
# =============================================================================
#
#  MIT-BIH CSV layout (shayanfazeli/heartbeat on Kaggle):
#    Columns 0–186  : 187 float samples (already min-max normalised to [0,1])
#    Column  187    : integer class label
#      0 = N  (Normal beat)          ← we label as 0
#      1 = S  (Supraventricular)     ┐
#      2 = V  (Ventricular PVC)      │ we label as 1 (Abnormal)
#      3 = F  (Fusion beat)          │
#      4 = Q  (Unclassifiable)       ┘
#
#  Class imbalance (approx):
#    Train : N≈72%, S≈3%, V≈21%, F≈0.7%, Q≈3%
#    → WeightedRandomSampler corrects this during training.
# =============================================================================

def load_mitbih_binary(data_dir: str):
    """Load MIT-BIH CSVs and return binary-labelled numpy arrays."""
    train_csv = os.path.join(data_dir, "mitbih_train.csv")
    test_csv  = os.path.join(data_dir, "mitbih_test.csv")

    if not os.path.isfile(train_csv) or not os.path.isfile(test_csv):
        sys.exit(
            "\n[ERROR] Dataset not found.\n"
            "  Expected files:\n"
            f"    {train_csv}\n"
            f"    {test_csv}\n"
            "  Download from Kaggle:\n"
            "    !kaggle datasets download -d shayanfazeli/heartbeat -p ./data --unzip\n"
        )

    df_tr = pd.read_csv(train_csv, header=None)
    df_te = pd.read_csv(test_csv,  header=None)

    X_train = df_tr.iloc[:, :187].values.astype(np.float32)
    y_train = df_tr.iloc[:,  187].values.astype(np.int64)

    X_test  = df_te.iloc[:, :187].values.astype(np.float32)
    y_test  = df_te.iloc[:,  187].values.astype(np.int64)

    # Binary remap: 0 → Normal, 1–4 → Abnormal
    y_train = (y_train > 0).astype(np.int64)
    y_test  = (y_test  > 0).astype(np.int64)

    print(f"\n[Data] Train samples : {len(X_train):,}  "
          f"| class dist: {dict(sorted(Counter(y_train).items()))}")
    print(f"[Data] Test  samples : {len(X_test):,}  "
          f"| class dist: {dict(sorted(Counter(y_test).items()))}")

    return X_train, y_train, X_test, y_test


# =============================================================================
#  SECTION 2 — Normalisation
# =============================================================================
#
#  The Kaggle CSV is already min-max normalised per beat, but we re-apply a
#  per-sample min-max pass to guarantee [0, 1] range — important if you later
#  swap in raw WFDB data or live ADC samples from the ARM PS.
# =============================================================================

def per_beat_minmax(X: np.ndarray) -> np.ndarray:
    """
    Normalise each beat independently to [0, 1].
    Shape: (N, 187)  →  (N, 187)
    """
    mins = X.min(axis=1, keepdims=True)
    maxs = X.max(axis=1, keepdims=True)
    denom = np.where((maxs - mins) < 1e-8, 1e-8, maxs - mins)
    return (X - mins) / denom


# =============================================================================
#  SECTION 3 — PyTorch Dataset & DataLoader helpers
# =============================================================================

class ECGDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        # Add channel dim → (N, 1, 187) for Conv1d
        self.X = torch.from_numpy(X[:, np.newaxis, :])   # float32
        self.y = torch.from_numpy(y.astype(np.float32))  # BCEWithLogitsLoss needs float

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def make_weighted_sampler(y: np.ndarray) -> WeightedRandomSampler:
    """
    Inverse-frequency weighting so the loader presents ~50/50 normal/abnormal
    mini-batches, counteracting MIT-BIH's ~4:1 class imbalance.
    """
    class_counts = np.bincount(y)                 # [count_normal, count_abnormal]
    sample_weights = 1.0 / class_counts[y]        # higher weight → rarer class
    return WeightedRandomSampler(
        weights     = torch.DoubleTensor(sample_weights),
        num_samples = len(sample_weights),
        replacement = True,
    )


# =============================================================================
#  SECTION 4 — Model Architecture
# =============================================================================
#
#  ECGAnomalyNet — 3-block 1D CNN
#  ─────────────────────────────────────────────────────────────────
#  Layer                  Kernel  Padding  Out shape      Params
#  ─────────────────────────────────────────────────────────────────
#  Input                                  (B,  1, 187)
#  Conv1d(1→16)  + BN + ReLU + MaxPool2   (B, 16,  93)    96 + 32
#  Conv1d(16→32) + BN + ReLU + MaxPool2   (B, 32,  46)  2,592 + 64
#  Conv1d(32→64) + BN + ReLU + MaxPool2   (B, 64,  23)  6,208 + 128
#  Flatten                                (B, 1472)
#  Linear(1472→64) + ReLU + Dropout(0.4) (B,   64)     94,272 + 64
#  Linear(64→1)   [raw logit]             (B,    1)         65
#  ─────────────────────────────────────────────────────────────────
#  Total trainable parameters ≈ 103,345
#  INT8 model size estimate   ≈ ~101 KB  (well within Zynq BRAM budget)
# =============================================================================

class ECGAnomalyNet(nn.Module):
    def __init__(self, dropout: float = 0.4):
        super().__init__()

        self.features = nn.Sequential(
            # ── Block 1 ──────────────────────────────────
            nn.Conv1d(in_channels=1,  out_channels=16,
                      kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),           # 187 → 93

            # ── Block 2 ──────────────────────────────────
            nn.Conv1d(in_channels=16, out_channels=32,
                      kernel_size=5, padding=2, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),           # 93 → 46

            # ── Block 3 ──────────────────────────────────
            nn.Conv1d(in_channels=32, out_channels=64,
                      kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=2),           # 46 → 23
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 23, 64, bias=True),    # 1472 → 64
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(64, 1, bias=True),           # 64 → logit (no sigmoid here)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x : (B, 1, 187)
        returns: (B,)  raw logit  [apply sigmoid for probability]
        """
        return self.classifier(self.features(x)).squeeze(1)

    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        """Convenience: returns sigmoid probability in [0,1]."""
        return torch.sigmoid(self.forward(x))


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# =============================================================================
#  SECTION 5 — Training & Validation Loops
# =============================================================================

def train_one_epoch(
    model     : nn.Module,
    loader    : DataLoader,
    optimizer : optim.Optimizer,
    criterion : nn.Module,
    device    : torch.device,
) -> tuple[float, float]:
    """Single training epoch. Returns (avg_loss, accuracy)."""
    model.train()
    running_loss = 0.0
    correct = 0
    total   = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(y_batch)
        preds  = (torch.sigmoid(logits) >= 0.5).long()
        correct += (preds == y_batch.long()).sum().item()
        total   += len(y_batch)

    return running_loss / total, correct / total


@torch.no_grad()
def validate(
    model     : nn.Module,
    loader    : DataLoader,
    criterion : nn.Module,
    device    : torch.device,
) -> tuple[float, np.ndarray, np.ndarray]:
    """Validation pass. Returns (avg_loss, y_true, y_pred)."""
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        running_loss += loss.item() * len(y_batch)

        preds = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.long().cpu().numpy())

    n = len(all_labels)
    return (
        running_loss / n,
        np.array(all_labels, dtype=np.int64),
        np.array(all_preds,  dtype=np.int64),
    )


# =============================================================================
#  SECTION 6 — Evaluation & Reporting
# =============================================================================

def print_metrics(y_true: np.ndarray, y_pred: np.ndarray, split: str = "Test"):
    """Print accuracy / precision / recall / F1 and full classification report."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n{'─'*50}")
    print(f"  {split} Results")
    print(f"{'─'*50}")
    print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  Precision : {prec:.4f}   [TP / (TP+FP)]")
    print(f"  Recall    : {rec:.4f}   [TP / (TP+FN)]  ← critical for clinical use")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"{'─'*50}")
    print(classification_report(
        y_true, y_pred,
        target_names=["Normal (0)", "Abnormal (1)"],
        digits=4,
    ))
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}


def plot_training_curves(
    train_losses : list[float],
    val_losses   : list[float],
    train_accs   : list[float],
    val_accs     : list[float],
    save_path    : str,
):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("ECGAnomalyNet — Training History", fontsize=14, fontweight="bold")
    epochs = range(1, len(train_losses) + 1)

    # Loss plot
    axes[0].plot(epochs, train_losses, label="Train", color="#2196F3")
    axes[0].plot(epochs, val_losses,   label="Val",   color="#F44336", linestyle="--")
    axes[0].set_title("Binary Cross-Entropy Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Accuracy plot
    axes[1].plot(epochs, [a*100 for a in train_accs], label="Train", color="#2196F3")
    axes[1].plot(epochs, [a*100 for a in val_accs],   label="Val",   color="#F44336", linestyle="--")
    axes[1].set_title("Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"[Plot] Saved → {save_path}")


def plot_confusion_matrix(y_true: np.ndarray, y_pred: np.ndarray):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Normal", "Abnormal"])
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title("Confusion Matrix — Test Set")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150)
    print("[Plot] Saved → confusion_matrix.png")


# =============================================================================
#  SECTION 7 — INT8 Quantization Export (post-training)
# =============================================================================
#
#  After training, call export_int8() to produce a quantized model ready for:
#    1. Weight extraction into Verilog ROM initialisation files (.mif / $readmemh)
#    2. Verification against the Verilog RTL inference core
#
#  This uses PyTorch's static post-training quantization (PTQ):
#    Calibration → fuse BN into Conv → quantize to INT8
# =============================================================================

def export_int8(
    model       : nn.Module,
    calib_loader: DataLoader,
    device      : torch.device,
    save_path   : str = "ecg_cnn_int8.pth",
):
    """
    Static post-training INT8 quantization using PyTorch's eager mode.
    The calibration set should be ~500–1000 representative samples.
    """
    import torch.ao.quantization as tq

    model_cpu = model.cpu().eval()

    # Fuse Conv + BN + ReLU into single quantized op
    model_fused = tq.fuse_modules(
        model_cpu,
        [
            ["features.0", "features.1", "features.2"],   # Block 1
            ["features.4", "features.5", "features.6"],   # Block 2
            ["features.8", "features.9", "features.10"],  # Block 3
        ],
    )

    model_fused.qconfig = tq.get_default_qconfig("fbgemm")  # x86; use "qnnpack" on ARM
    tq.prepare(model_fused, inplace=True)

    # Calibration pass — collect activation statistics
    print("[INT8] Running calibration pass …")
    model_fused.eval()
    with torch.no_grad():
        for i, (X_batch, _) in enumerate(calib_loader):
            model_fused(X_batch.cpu())
            if i >= 8:           # ~1024 samples is enough for calibration
                break

    tq.convert(model_fused, inplace=True)
    torch.save(model_fused.state_dict(), save_path)
    print(f"[INT8] Quantized model saved → {save_path}")
    print("[INT8] Next: run extract_weights.py to dump .mif files for Verilog ROMs.")
    return model_fused


# =============================================================================
#  SECTION 8 — Main Entry Point
# =============================================================================

def main():
    cfg    = CFG
    device = torch.device(cfg["device"])

    # ── 1. Load & preprocess data ─────────────────────────────────────────────
    X_train, y_train, X_test, y_test = load_mitbih_binary(cfg["data_dir"])
    X_train = per_beat_minmax(X_train)
    X_test  = per_beat_minmax(X_test)

    # ── 2. Train / validation split (stratified) ──────────────────────────────
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train,
        test_size   = cfg["val_split"],
        random_state= SEED,
        stratify    = y_train,
    )
    print(f"\n[Split] Train: {len(X_tr):,}  |  Val: {len(X_val):,}  |  Test: {len(X_test):,}")

    # ── 3. Datasets & DataLoaders ─────────────────────────────────────────────
    train_ds = ECGDataset(X_tr,   y_tr)
    val_ds   = ECGDataset(X_val,  y_val)
    test_ds  = ECGDataset(X_test, y_test)

    sampler  = make_weighted_sampler(y_tr)

    train_loader = DataLoader(
        train_ds,
        batch_size  = cfg["batch_size"],
        sampler     = sampler,
        num_workers = cfg["num_workers"],
        pin_memory  = (cfg["device"] == "cuda"),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size  = cfg["batch_size"],
        shuffle     = False,
        num_workers = cfg["num_workers"],
    )
    test_loader = DataLoader(
        test_ds,
        batch_size  = cfg["batch_size"],
        shuffle     = False,
        num_workers = cfg["num_workers"],
    )

    # ── 4. Model / loss / optimiser / scheduler ───────────────────────────────
    model     = ECGAnomalyNet(dropout=cfg["dropout"]).to(device)
    criterion = nn.BCEWithLogitsLoss()                 # numerically stable; no sigmoid in model
    optimizer = optim.Adam(
        model.parameters(),
        lr           = cfg["lr"],
        weight_decay = cfg["weight_decay"],
    )
    # Cosine annealing: smoothly decays LR from lr → eta_min over all epochs
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg["epochs"], eta_min=1e-5
    )

    total_params = count_parameters(model)
    print(f"\n[Model] Trainable parameters : {total_params:,}")
    print(f"[Model] Estimated FP32 size  : {total_params * 4 / 1024:.1f} KB")
    print(f"[Model] Estimated INT8 size  : {total_params * 1 / 1024:.1f} KB")

    # ── 5. Training loop ──────────────────────────────────────────────────────
    train_losses, val_losses = [], []
    train_accs,   val_accs   = [], []
    best_val_loss = float("inf")

    print(f"\n[Train] Starting {cfg['epochs']} epochs …\n")

    for epoch in range(1, cfg["epochs"] + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        vl_loss, y_vt, y_vp = validate(model, val_loader, criterion, device)
        vl_acc = accuracy_score(y_vt, y_vp)
        scheduler.step()

        train_losses.append(tr_loss);  val_losses.append(vl_loss)
        train_accs.append(tr_acc);     val_accs.append(vl_acc)

        # Save best checkpoint
        ckpt_tag = ""
        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            torch.save(model.state_dict(), cfg["best_ckpt"])
            ckpt_tag = "  ← best ✓"

        print(
            f"Epoch {epoch:3d}/{cfg['epochs']}  |  "
            f"Train  loss={tr_loss:.4f}  acc={tr_acc*100:.2f}%  |  "
            f"Val    loss={vl_loss:.4f}  acc={vl_acc*100:.2f}%"
            f"{ckpt_tag}"
        )

    # Save final weights (useful for continued fine-tuning)
    torch.save(model.state_dict(), cfg["final_ckpt"])
    print(f"\n[Save] Final weights  → {cfg['final_ckpt']}")
    print(f"[Save] Best  weights  → {cfg['best_ckpt']}")

    # ── 6. Final test evaluation (load best checkpoint) ───────────────────────
    print("\n[Eval] Loading best checkpoint for test evaluation …")
    model.load_state_dict(torch.load(cfg["best_ckpt"], map_location=device))
    _, y_true, y_pred = validate(model, test_loader, criterion, device)

    metrics = print_metrics(y_true, y_pred, split="Test (best checkpoint)")

    # ── 7. Plots ──────────────────────────────────────────────────────────────
    plot_training_curves(train_losses, val_losses, train_accs, val_accs, cfg["plot_path"])
    plot_confusion_matrix(y_true, y_pred)

    # ── 8. INT8 export ────────────────────────────────────────────────────────
    print("\n[INT8] Starting post-training quantization …")
    try:
        export_int8(model, train_loader, device, save_path="ecg_cnn_int8.pth")
    except Exception as e:
        print(f"[INT8] Skipped (optional step): {e}")

    # ── 9. Summary ────────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("  TRAINING COMPLETE")
    print("="*60)
    print(f"  Test Accuracy  : {metrics['accuracy']*100:.2f}%")
    print(f"  Test F1-Score  : {metrics['f1']:.4f}")
    print(f"  Test Recall    : {metrics['recall']:.4f}  (sensitivity)")
    print(f"  Saved outputs  :")
    print(f"    {cfg['best_ckpt']}        ← use for RTL weight extraction")
    print(f"    ecg_cnn_int8.pth       ← INT8 quantized (FPGA-ready)")
    print(f"    {cfg['plot_path']}  ← training curves")
    print(f"    confusion_matrix.png   ← test set confusion matrix")
    print("="*60)
    print("\n[Next] Run weight_extractor.py to dump INT8 weights as")
    print("       Verilog $readmemh hex files for BRAM initialisation.")


# =============================================================================
#  Run
# =============================================================================
if __name__ == "__main__":
    main()

In [ ]:
%%writefile weight_extractor.py
# =============================================================================
#  Smart Hospital Edge AI — FPGA Weight Extractor
#  ECGAnomalyNet  →  INT8 HEX files  ($readmemh-compatible)
#
#  Inputs  : ecg_cnn_best.pth   (float32 checkpoint)      ← primary path
#            ecg_cnn_int8.pth   (PyTorch PTQ checkpoint)  ← optional
#  Outputs : hex/conv1_weights.hex   hex/conv1_bias.hex
#            hex/conv2_weights.hex   hex/conv2_bias.hex
#            hex/conv3_weights.hex   hex/conv3_bias.hex
#            hex/fc1_weights.hex     hex/fc1_bias.hex
#            hex/fc2_weights.hex     hex/fc2_bias.hex
#            hex/weights_manifest.json
#            hex/weights_pkg.vh      ← Verilog parameter include
#
#  Quantization scheme used here:
#    Conv layers : Symmetric per-output-channel INT8
#                  BatchNorm folded INTO Conv weights before quantizing
#    FC   layers : Symmetric per-tensor INT8
#    Biases      : INT32 (quantized with same scale as conv/fc weights;
#                  hardware accumulates as: acc_int32 + bias_int32)
#
#  $readmemh format:
#    INT8  values → 2 hex chars per line  (e.g. "7f", "ff" = -1)
#    INT32 values → 8 hex chars per line  (e.g. "000001ff")
#    All negative values stored as two's complement.
#
#  RTL memory address formula (Conv1d weight ROM):
#    addr = oc * (Cin * K) + ic * K + k
#    where oc ∈ [0, Cout), ic ∈ [0, Cin), k ∈ [0, K)
#    Total words = Cout * Cin * K
#
#  RTL memory address formula (FC weight ROM):
#    addr = row * Fin + col
#    where row ∈ [0, Fout), col ∈ [0, Fin)
#    Total words = Fout * Fin
# =============================================================================
#
#  COLAB QUICK START:
#    !python weight_extractor.py
#    # Produces hex/ directory with all output files.
#    # Verify with:  !python weight_extractor.py --verify
# =============================================================================

import os
import sys
import json
import struct
import argparse
import numpy as np

import torch
import torch.nn as nn


# =============================================================================
#  SECTION 0 — Config
# =============================================================================
CFG = {
    "float_ckpt"  : "ecg_cnn_best.pth",
    "int8_ckpt"   : "ecg_cnn_int8.pth",
    "output_dir"  : "hex",
    "use_int8_ckpt": False,     # set True to load PyTorch PTQ model instead
    "int8_clip"   : 127,        # max positive value for symmetric INT8
    "int8_min"    : -128,       # min value (allows full [-128, 127] range)
}


# =============================================================================
#  SECTION 1 — Model definition (must match training pipeline exactly)
# =============================================================================
class ECGAnomalyNet(nn.Module):
    """Mirror of ecg_training_pipeline.py — do not modify."""
    def __init__(self, dropout: float = 0.4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1,  16, kernel_size=5, padding=2, bias=False),  # [0]
            nn.BatchNorm1d(16),                                        # [1]
            nn.ReLU(inplace=True),                                     # [2]
            nn.MaxPool1d(kernel_size=2),                               # [3]

            nn.Conv1d(16, 32, kernel_size=5, padding=2, bias=False),  # [4]
            nn.BatchNorm1d(32),                                        # [5]
            nn.ReLU(inplace=True),                                     # [6]
            nn.MaxPool1d(kernel_size=2),                               # [7]

            nn.Conv1d(32, 64, kernel_size=3, padding=1, bias=False),  # [8]
            nn.BatchNorm1d(64),                                        # [9]
            nn.ReLU(inplace=True),                                     # [10]
            nn.MaxPool1d(kernel_size=2),                               # [11]
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                                              # [0]
            nn.Linear(64 * 23, 64, bias=True),                        # [1]
            nn.ReLU(inplace=True),                                     # [2]
            nn.Dropout(p=dropout),                                     # [3]
            nn.Linear(64, 1, bias=True),                               # [4]
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)


# =============================================================================
#  SECTION 2 — BatchNorm Folding
# =============================================================================
#
#  For each Conv1d (bias=False) followed by BatchNorm1d:
#
#    Standard BN formula during inference:
#      y = gamma * (x - running_mean) / sqrt(running_var + eps) + beta
#
#    Fusing BN into Conv means rewriting:
#      y = W_folded * x + b_folded
#    where for each output channel oc:
#
#      W_folded[oc, :, :] = W[oc, :, :] * (gamma[oc] / sqrt(running_var[oc] + eps))
#      b_folded[oc]        = beta[oc]  -  gamma[oc] * running_mean[oc]
#                                         / sqrt(running_var[oc] + eps)
#
#  After folding, each Conv block is equivalent to a biased Conv1d with no BN.
# =============================================================================

def fold_bn_into_conv(
    conv_w      : np.ndarray,   # (Cout, Cin, K)  float32
    bn_gamma    : np.ndarray,   # (Cout,)
    bn_beta     : np.ndarray,   # (Cout,)
    bn_mean     : np.ndarray,   # (Cout,)
    bn_var      : np.ndarray,   # (Cout,)
    bn_eps      : float = 1e-5,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Returns (W_folded, b_folded) as float32 numpy arrays.
    W_folded shape: (Cout, Cin, K)
    b_folded shape: (Cout,)
    """
    # Per-channel BN scale factor, shape (Cout,)
    bn_std   = np.sqrt(bn_var + bn_eps)          # (Cout,)
    bn_scale = bn_gamma / bn_std                  # (Cout,)

    # Broadcast over (Cin, K) dimensions
    W_folded = conv_w * bn_scale[:, np.newaxis, np.newaxis]  # (Cout, Cin, K)
    b_folded = bn_beta - bn_gamma * bn_mean / bn_std          # (Cout,)

    return W_folded.astype(np.float32), b_folded.astype(np.float32)


# =============================================================================
#  SECTION 3 — INT8 Quantization Helpers
# =============================================================================
#
#  We use SYMMETRIC quantization (zero_point = 0):
#    scale   = max(|W|) / INT8_MAX
#    W_int8  = clamp( round(W / scale), INT8_MIN, INT8_MAX )
#
#  Per-channel for Conv (one scale per output filter):
#    scale[oc] = max(|W[oc, :, :]|) / INT8_MAX
#
#  Per-tensor for FC:
#    scale = max(|W|) / INT8_MAX
#
#  Bias is quantized using the WEIGHT scale only (no input scale at this stage).
#  The RTL must later apply: output = (acc_int32 + bias_int32) * (w_scale * x_scale)
# =============================================================================

def quantize_per_channel(
    W       : np.ndarray,   # (Cout, ...)  float32
    int8_max: int = 127,
    int8_min: int = -128,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Symmetric per-output-channel INT8 quantization.
    Returns (W_int8: int8 ndarray, scales: float32 ndarray shape (Cout,)).
    """
    Cout = W.shape[0]
    # Max absolute value per output channel, over all other dims
    max_abs = np.abs(W.reshape(Cout, -1)).max(axis=1)          # (Cout,)
    max_abs = np.where(max_abs < 1e-8, 1e-8, max_abs)          # avoid div/0
    scales  = max_abs / float(int8_max)                          # (Cout,)

    # Broadcast scale for division: (Cout, 1, 1) for (Cout, Cin, K)
    extra_dims = tuple(1 for _ in W.shape[1:])
    W_scaled = W / scales.reshape((Cout,) + extra_dims)
    W_int8   = np.clip(np.round(W_scaled), int8_min, int8_max).astype(np.int8)

    return W_int8, scales.astype(np.float32)


def quantize_per_tensor(
    W       : np.ndarray,   # any shape, float32
    int8_max: int = 127,
    int8_min: int = -128,
) -> tuple[np.ndarray, float]:
    """
    Symmetric per-tensor INT8 quantization.
    Returns (W_int8: int8 ndarray, scale: float).
    """
    max_abs = float(np.abs(W).max())
    if max_abs < 1e-8:
        max_abs = 1e-8
    scale   = max_abs / float(int8_max)
    W_int8  = np.clip(np.round(W / scale), int8_min, int8_max).astype(np.int8)
    return W_int8, scale


def quantize_bias_int32(
    bias    : np.ndarray,   # (Cout,)  float32
    scales  : np.ndarray,   # per-channel or broadcast scalar, float32
) -> np.ndarray:
    """
    Quantize bias to INT32 using weight scale.
    bias_int32[oc] = round(bias[oc] / scales[oc])
    Hardware accumulates: acc_int32 + bias_int32  before scaling.
    """
    bias_int32 = np.round(bias / scales).astype(np.int32)
    return bias_int32


# =============================================================================
#  SECTION 4 — HEX File Writers ($readmemh-compatible)
# =============================================================================
#
#  INT8  → 2 hex chars (e.g.  127 → "7f",  -1 → "ff",  -128 → "80")
#  INT32 → 8 hex chars (e.g.  255 → "000000ff",  -1 → "ffffffff")
#  Each value on its own line, lower-case hex, no "0x" prefix.
#
#  A header comment block at the top records: shape, dtype, order, scales.
# =============================================================================

def int8_to_hex(v: int) -> str:
    """Two's-complement 2-char hex for a signed 8-bit integer."""
    return f"{v & 0xFF:02x}"


def int32_to_hex(v: int) -> str:
    """Two's-complement 8-char hex for a signed 32-bit integer."""
    return f"{v & 0xFFFFFFFF:08x}"


def write_hex_file(
    path     : str,
    data_int : np.ndarray,      # flattened int8 or int32 array
    header   : list[str],       # list of comment lines (without leading //)
    dtype    : str = "int8",    # "int8" or "int32"
):
    """
    Write a $readmemh-compatible hex file.
    Comments use // prefix (supported by most simulators).
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    flat = data_int.flatten()

    with open(path, "w") as f:
        for line in header:
            f.write(f"// {line}\n")
        f.write("//\n")

        if dtype == "int8":
            for val in flat:
                f.write(int8_to_hex(int(val)) + "\n")
        elif dtype == "int32":
            for val in flat:
                f.write(int32_to_hex(int(val)) + "\n")
        else:
            raise ValueError(f"Unsupported dtype: {dtype}")

    print(f"  [HEX] {os.path.basename(path):30s}  {len(flat):>8,} entries  ({dtype})")


# =============================================================================
#  SECTION 5 — Layer Extraction (Float Model + BN Fold Path)
# =============================================================================

def extract_float_path(model: ECGAnomalyNet) -> dict:
    """
    Primary extraction path:
      1. Read Conv + BN weights from float model
      2. Fold BN into Conv
      3. Quantize Conv to INT8 (per-channel) and FC to INT8 (per-tensor)
      4. Quantize biases to INT32

    Returns a dict of LayerInfo objects ready for hex writing.
    """
    sd = {k: v.detach().cpu().numpy()
          for k, v in model.state_dict().items()}

    # ── Helper to read BN params ──────────────────────────────────────────────
    def get_bn(prefix):
        return (
            sd[f"{prefix}.weight"],        # gamma
            sd[f"{prefix}.bias"],          # beta
            sd[f"{prefix}.running_mean"],
            sd[f"{prefix}.running_var"],
        )

    layers = {}

    # ── Conv blocks 1–3 ──────────────────────────────────────────────────────
    conv_specs = [
        ("conv1", "features.0", "features.1"),   # Conv1d(1,16,k=5)  + BN
        ("conv2", "features.4", "features.5"),   # Conv1d(16,32,k=5) + BN
        ("conv3", "features.8", "features.9"),   # Conv1d(32,64,k=3) + BN
    ]

    for name, conv_key, bn_key in conv_specs:
        W    = sd[f"{conv_key}.weight"]          # (Cout, Cin, K)
        gamma, beta, mean, var = get_bn(bn_key)

        W_folded, b_folded = fold_bn_into_conv(W, gamma, beta, mean, var)

        W_int8, w_scales = quantize_per_channel(W_folded)
        b_int32          = quantize_bias_int32(b_folded, w_scales)

        layers[name] = {
            "W_int8"  : W_int8,
            "b_int32" : b_int32,
            "w_scales": w_scales,
            "W_shape" : W_int8.shape,        # (Cout, Cin, K)
            "b_shape" : b_int32.shape,       # (Cout,)
            "type"    : "conv",
        }

    # ── FC layers ─────────────────────────────────────────────────────────────
    # classifier[1] = Linear(1472, 64),  classifier[4] = Linear(64, 1)
    fc_specs = [
        ("fc1", "classifier.1"),
        ("fc2", "classifier.4"),
    ]

    for name, key in fc_specs:
        W = sd[f"{key}.weight"]              # (Fout, Fin)
        b = sd[f"{key}.bias"]                # (Fout,)

        W_int8, w_scale = quantize_per_tensor(W)
        b_int32         = quantize_bias_int32(b, np.full(b.shape, w_scale, dtype=np.float32))

        layers[name] = {
            "W_int8"  : W_int8,
            "b_int32" : b_int32,
            "w_scales": np.array([w_scale], dtype=np.float32),
            "W_shape" : W_int8.shape,        # (Fout, Fin)
            "b_shape" : b_int32.shape,       # (Fout,)
            "type"    : "fc",
        }

    return layers


# =============================================================================
#  SECTION 6 — Layer Extraction (PyTorch PTQ Model Path)
# =============================================================================

def extract_ptq_path(ptq_model) -> dict:
    """
    Optional extraction path: loads a model already quantized via
    torch.ao.quantization.  Extracts int_repr() tensors directly.

    Note: PyTorch PTQ uses asymmetric quantization by default (non-zero
    zero_point), whereas our float path uses symmetric.  The manifest
    records both scale and zero_point for each layer.
    """
    import torch.ao.quantization as tq

    layers = {}
    names  = ["conv1", "conv2", "conv3"]
    # After fuse_modules + convert, conv blocks are at features[0], [4], [8]
    feat_indices = [0, 4, 8]

    for name, idx in zip(names, feat_indices):
        q_conv = ptq_model.features[idx]
        W_q    = q_conv.weight()                              # quantized tensor
        W_int8 = W_q.int_repr().numpy().astype(np.int8)      # (Cout, Cin, K)

        try:
            w_scales = W_q.q_per_channel_scales().numpy()    # per-channel
            w_zp     = W_q.q_per_channel_zero_points().numpy()
        except Exception:
            w_scales = np.array([W_q.q_scale()])
            w_zp     = np.array([W_q.q_zero_point()])

        b_raw  = q_conv.bias().detach().cpu().numpy()
        b_int32 = np.round(b_raw / (w_scales if w_scales.ndim > 0 else w_scales[0])).astype(np.int32)

        layers[name] = {
            "W_int8"  : W_int8,
            "b_int32" : b_int32,
            "w_scales": w_scales,
            "w_zp"    : w_zp,
            "W_shape" : W_int8.shape,
            "b_shape" : b_int32.shape,
            "type"    : "conv",
        }

    # FC layers
    for name, mod in [("fc1", ptq_model.classifier[1]),
                      ("fc2", ptq_model.classifier[4])]:
        W_q    = mod.weight()
        W_int8 = W_q.int_repr().numpy().astype(np.int8)
        scale  = W_q.q_scale()
        zp     = W_q.q_zero_point()
        b_raw  = mod.bias().detach().cpu().numpy()
        b_int32 = np.round(b_raw / scale).astype(np.int32)

        layers[name] = {
            "W_int8"  : W_int8,
            "b_int32" : b_int32,
            "w_scales": np.array([scale], dtype=np.float32),
            "w_zp"    : np.array([zp]),
            "W_shape" : W_int8.shape,
            "b_shape" : b_int32.shape,
            "type"    : "fc",
        }

    return layers


# =============================================================================
#  SECTION 7 — Write All HEX Files
# =============================================================================

def write_all_hex(layers: dict, out_dir: str):
    """
    Write one hex file per weight tensor and one per bias tensor.
    Files are placed in out_dir/.

    Conv weight memory layout (RTL address formula):
      addr = oc * (Cin * K) + ic * K + k
      → store in row-major order: iterate oc, then ic, then k

    FC weight memory layout:
      addr = row * Fin + col
      → standard C row-major order
    """
    os.makedirs(out_dir, exist_ok=True)

    layer_order = ["conv1", "conv2", "conv3", "fc1", "fc2"]

    for lname in layer_order:
        info = layers[lname]
        W    = info["W_int8"]
        B    = info["b_int32"]
        sc   = info["w_scales"]
        shape = info["W_shape"]

        # ── Weights ───────────────────────────────────────────────────────────
        if info["type"] == "conv":
            Cout, Cin, K = shape
            order_note = (
                f"Memory order: addr = oc*(Cin*K) + ic*K + k",
                f"              oc in [0,{Cout}), ic in [0,{Cin}), k in [0,{K})",
                f"Total words  : {Cout*Cin*K}",
            )
        else:
            Fout, Fin = shape
            order_note = (
                f"Memory order: addr = row*Fin + col",
                f"              row in [0,{Fout}), col in [0,{Fin})",
                f"Total words  : {Fout*Fin}",
            )

        scale_note = (
            "Per-channel scales (oc=0 first):"
            if info["type"] == "conv" and len(sc) > 1
            else f"Scale (per-tensor): {sc[0]:.8e}"
        )
        scale_lines = [scale_note]
        if info["type"] == "conv" and len(sc) > 1:
            scale_lines += [f"  oc={i:3d}: {s:.8e}" for i, s in enumerate(sc)]

        header = [
            f"Layer       : {lname}",
            f"Tensor      : weights",
            f"Shape       : {shape}",
            f"Dtype       : INT8  (symmetric, zero_point=0)",
            *order_note,
            *scale_lines,
            f"Use in RTL  : $readmemh(\"{lname}_weights.hex\", rom_{lname}_w);",
        ]

        write_hex_file(
            path     = os.path.join(out_dir, f"{lname}_weights.hex"),
            data_int = W,          # already in (Cout, Cin, K) or (Fout, Fin) order
            header   = header,
            dtype    = "int8",
        )

        # ── Biases ────────────────────────────────────────────────────────────
        b_header = [
            f"Layer       : {lname}",
            f"Tensor      : bias",
            f"Shape       : {B.shape}",
            f"Dtype       : INT32 (quantized with weight scale)",
            f"Formula     : bias_int32 = round(bias_float / w_scale[oc])",
            f"RTL usage   : acc_int32 + bias_int32  (before output re-scaling)",
            f"Use in RTL  : $readmemh(\"{lname}_bias.hex\", rom_{lname}_b);",
        ]

        write_hex_file(
            path     = os.path.join(out_dir, f"{lname}_bias.hex"),
            data_int = B,
            header   = b_header,
            dtype    = "int32",
        )


# =============================================================================
#  SECTION 8 — Metadata: JSON Manifest + Verilog .vh Parameter File
# =============================================================================

def write_manifest(layers: dict, out_dir: str):
    """
    JSON manifest: shapes, quantization scales/zero_points, word counts.
    Used by Verilog RTL generator and PS-side verification script.
    """
    manifest = {}
    for lname, info in layers.items():
        sc = info["w_scales"]
        manifest[lname] = {
            "type"           : info["type"],
            "weight_shape"   : list(info["W_shape"]),
            "bias_shape"     : list(info["b_shape"]),
            "weight_words"   : int(np.prod(info["W_shape"])),
            "bias_words"     : int(np.prod(info["b_shape"])),
            "quant_scheme"   : "symmetric_per_channel" if len(sc) > 1 else "symmetric_per_tensor",
            "zero_point"     : 0,
            "w_scale_min"    : float(sc.min()),
            "w_scale_max"    : float(sc.max()),
            "w_scales"       : sc.tolist(),
        }

    path = os.path.join(out_dir, "weights_manifest.json")
    with open(path, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"  [JSON] {os.path.basename(path)}")
    return manifest


def write_verilog_pkg(layers: dict, manifest: dict, out_dir: str):
    """
    Verilog parameter include file (weights_pkg.vh).
    Provides ROM depth parameters and per-layer scale factors as
    Q8.24 fixed-point localparams for use in the RTL inference core.

    RTL usage:
      `include "weights_pkg.vh"
      logic signed [7:0] rom_conv1_w [0:CONV1_W_DEPTH-1];
      initial $readmemh("conv1_weights.hex", rom_conv1_w);
    """
    lines = [
        "// =================================================================",
        "//  weights_pkg.vh — Auto-generated by weight_extractor.py",
        "//  DO NOT EDIT MANUALLY. Re-run weight_extractor.py to regenerate.",
        "// =================================================================",
        "",
        "`ifndef WEIGHTS_PKG_VH",
        "`define WEIGHTS_PKG_VH",
        "",
    ]

    layer_order = ["conv1", "conv2", "conv3", "fc1", "fc2"]

    for lname in layer_order:
        info = manifest[lname]
        sc   = layers[lname]["w_scales"]
        prefix = lname.upper()

        lines += [
            f"// ── {lname} ──────────────────────────────────────────────────────",
            f"localparam integer {prefix}_W_DEPTH = {info['weight_words']};   "
            f"// weight ROM depth",
            f"localparam integer {prefix}_B_DEPTH = {info['bias_words']};    "
            f"// bias ROM depth",
        ]

        # Embed per-channel scales as Q8.24 fixed-point (32-bit)
        # Q8.24: 8 integer bits, 24 fractional bits
        # scale_fixed = round(scale * 2^24)
        q_shift = 24
        scale_fixed = np.round(sc * (2 ** q_shift)).astype(np.int64)

        if len(sc) == 1:
            lines.append(
                f"localparam signed [31:0] {prefix}_SCALE = 32'h{scale_fixed[0] & 0xFFFFFFFF:08x}; "
                f"// Q8.24 ≈ {sc[0]:.6e}"
            )
        else:
            lines.append(
                f"// Per-channel Q8.24 scales ({len(sc)} channels)"
            )
            # Pack into a flat parameter array hint (full arrays need SystemVerilog packages
            # or unpacked initialisation; we emit individual params for compatibility)
            for i, (sf, sv) in enumerate(zip(scale_fixed, sc)):
                lines.append(
                    f"localparam signed [31:0] {prefix}_SCALE_{i:03d} = "
                    f"32'h{sf & 0xFFFFFFFF:08x}; // Q8.24 ≈ {sv:.6e}"
                )

        lines.append("")

    lines += [
        "// ── Architecture constants ──────────────────────────────────────────",
        "localparam integer INPUT_LEN  = 187;",
        "localparam integer INPUT_CH   = 1;",
        "localparam integer CONV1_OUT  = 16;",
        "localparam integer CONV2_OUT  = 32;",
        "localparam integer CONV3_OUT  = 64;",
        "localparam integer FC1_IN     = 1472;",
        "localparam integer FC1_OUT    = 64;",
        "localparam integer FC2_OUT    = 1;",
        "",
        "`endif // WEIGHTS_PKG_VH",
    ]

    path = os.path.join(out_dir, "weights_pkg.vh")
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")
    print(f"  [VH]   {os.path.basename(path)}")


# =============================================================================
#  SECTION 9 — Numerical Verification
# =============================================================================
#
#  Runs a forward pass through the INT8 model (float arithmetic, but uses
#  only integer weight values * scale) and compares against the float model.
#  Reports per-layer cosine similarity to detect any extraction errors.
# =============================================================================

def verify_extraction(float_model: ECGAnomalyNet, layers: dict):
    """
    Spot-check extraction correctness by comparing dequantized weights
    against the BN-folded float weights.  Reports cosine similarity per
    layer — values > 0.999 indicate correct extraction.
    """
    print("\n[Verify] Cosine similarity: dequantized INT8 vs BN-folded float")
    print("         (Target: > 0.999 for all layers)")
    print(f"  {'Layer':8s}  {'Shape':20s}  {'Cosine Sim':>12s}  {'Max Err':>12s}")
    print(f"  {'─'*8}  {'─'*20}  {'─'*12}  {'─'*12}")

    sd = {k: v.detach().cpu().numpy()
          for k, v in float_model.state_dict().items()}

    conv_specs = [
        ("conv1", "features.0", "features.1"),
        ("conv2", "features.4", "features.5"),
        ("conv3", "features.8", "features.9"),
    ]
    fc_specs = [
        ("fc1", "classifier.1"),
        ("fc2", "classifier.4"),
    ]

    all_pass = True

    for name, conv_key, bn_key in conv_specs:
        W = sd[f"{conv_key}.weight"]
        gamma, beta, mean, var = (
            sd[f"{bn_key}.weight"], sd[f"{bn_key}.bias"],
            sd[f"{bn_key}.running_mean"], sd[f"{bn_key}.running_var"],
        )
        W_ref, _ = fold_bn_into_conv(W, gamma, beta, mean, var)

        # Dequantize extracted INT8 back to float
        W_i8 = layers[name]["W_int8"].astype(np.float32)   # (Cout, Cin, K)
        sc   = layers[name]["w_scales"]                      # (Cout,)
        W_deq = W_i8 * sc[:, np.newaxis, np.newaxis]

        cos_sim = float(
            np.dot(W_ref.flatten(), W_deq.flatten()) /
            (np.linalg.norm(W_ref) * np.linalg.norm(W_deq) + 1e-12)
        )
        max_err = float(np.abs(W_ref - W_deq).max())
        ok = "✓" if cos_sim > 0.999 else "✗ FAIL"
        if cos_sim <= 0.999:
            all_pass = False
        print(f"  {name:8s}  {str(W_ref.shape):20s}  {cos_sim:12.6f}  {max_err:12.6e}  {ok}")

    for name, key in fc_specs:
        W_ref = sd[f"{key}.weight"].astype(np.float32)
        W_i8  = layers[name]["W_int8"].astype(np.float32)
        sc    = float(layers[name]["w_scales"][0])
        W_deq = W_i8 * sc

        cos_sim = float(
            np.dot(W_ref.flatten(), W_deq.flatten()) /
            (np.linalg.norm(W_ref) * np.linalg.norm(W_deq) + 1e-12)
        )
        max_err = float(np.abs(W_ref - W_deq).max())
        ok = "✓" if cos_sim > 0.999 else "✗ FAIL"
        if cos_sim <= 0.999:
            all_pass = False
        print(f"  {name:8s}  {str(W_ref.shape):20s}  {cos_sim:12.6f}  {max_err:12.6e}  {ok}")

    print()
    if all_pass:
        print("[Verify] All layers passed  ✓  Hex files are correct.")
    else:
        print("[Verify] Some layers FAILED ✗  Check quantization scheme.")
    return all_pass


# =============================================================================
#  SECTION 10 — Summary printer
# =============================================================================

def print_summary(layers: dict, out_dir: str):
    print("\n" + "=" * 68)
    print("  WEIGHT EXTRACTION SUMMARY")
    print("=" * 68)
    print(f"  {'Layer':8s}  {'Weight shape':20s}  {'Words':>8s}  "
          f"{'Scale range':>26s}")
    print(f"  {'─'*8}  {'─'*20}  {'─'*8}  {'─'*26}")

    total_words = 0
    for name in ["conv1", "conv2", "conv3", "fc1", "fc2"]:
        info = layers[name]
        sc   = info["w_scales"]
        words = int(np.prod(info["W_shape"]))
        total_words += words
        sc_str = (f"[{sc.min():.3e} … {sc.max():.3e}]"
                  if len(sc) > 1 else f"{sc[0]:.6e}")
        print(f"  {name:8s}  {str(info['W_shape']):20s}  {words:>8,}  {sc_str:>26s}")

    print(f"  {'─'*8}  {'─'*20}  {'─'*8}  {'─'*26}")
    print(f"  {'TOTAL':8s}  {'':20s}  {total_words:>8,}  INT8 → {total_words/1024:.1f} KB")
    print("=" * 68)
    print(f"\n  Output directory : {os.path.abspath(out_dir)}/")
    print(f"  Files written    :")
    for f in sorted(os.listdir(out_dir)):
        size = os.path.getsize(os.path.join(out_dir, f))
        print(f"    {f:35s}  {size:>8,} bytes")
    print()
    print("  Next steps:")
    print("    1. Copy hex/ into your Vivado/Vitis project source tree")
    print("    2. `include \"weights_pkg.vh\" in your RTL inference core")
    print("    3. Use $readmemh() in initial blocks to load each ROM")
    print("    4. Feed conv*_SCALE values into your output requantization logic")
    print("=" * 68)


# =============================================================================
#  SECTION 11 — Main
# =============================================================================

def main():
    parser = argparse.ArgumentParser(
        description="Extract ECGAnomalyNet weights to $readmemh hex files"
    )
    parser.add_argument("--float-ckpt",    default=CFG["float_ckpt"])
    parser.add_argument("--int8-ckpt",     default=CFG["int8_ckpt"])
    parser.add_argument("--output-dir",    default=CFG["output_dir"])
    parser.add_argument("--use-ptq",       action="store_true",
                        help="Load from PyTorch PTQ model instead of float model")
    parser.add_argument("--verify",        action="store_true",
                        help="Run cosine-similarity verification after extraction")
    args = parser.parse_args()

    out_dir = args.output_dir
    os.makedirs(out_dir, exist_ok=True)

    # ── Load model ────────────────────────────────────────────────────────────
    if args.use_ptq:
        # ── PTQ path: reconstruct quantized model skeleton, load state_dict ──
        print(f"\n[Load] PyTorch PTQ model from {args.int8_ckpt} …")
        if not os.path.isfile(args.int8_ckpt):
            sys.exit(f"[ERROR] File not found: {args.int8_ckpt}\n"
                     "        Run ecg_training_pipeline.py first.")
        try:
            import torch.ao.quantization as tq
            base = ECGAnomalyNet().cpu().eval()
            fused = tq.fuse_modules(base, [
                ["features.0", "features.1", "features.2"],
                ["features.4", "features.5", "features.6"],
                ["features.8", "features.9", "features.10"],
            ])
            fused.qconfig = tq.get_default_qconfig("fbgemm")
            tq.prepare(fused, inplace=True)
            tq.convert(fused, inplace=True)
            fused.load_state_dict(
                torch.load(args.int8_ckpt, map_location="cpu"), strict=False
            )
            ptq_model = fused.eval()
            layers = extract_ptq_path(ptq_model)
            float_model = None
            print("[Load] PTQ model loaded OK")
        except Exception as e:
            sys.exit(f"[ERROR] Failed to load PTQ model: {e}\n"
                     "        Try without --use-ptq to use float model path.")

    else:
        # ── Float path (primary): load float checkpoint ────────────────────
        print(f"\n[Load] Float model from {args.float_ckpt} …")
        if not os.path.isfile(args.float_ckpt):
            sys.exit(f"[ERROR] File not found: {args.float_ckpt}\n"
                     "        Run ecg_training_pipeline.py first.")
        float_model = ECGAnomalyNet().cpu().eval()
        float_model.load_state_dict(
            torch.load(args.float_ckpt, map_location="cpu")
        )
        print("[Load] Float model loaded OK")
        layers = extract_float_path(float_model)

    # ── Write hex files ───────────────────────────────────────────────────────
    print(f"\n[Write] Generating hex files → {out_dir}/")
    write_all_hex(layers, out_dir)

    # ── Write metadata ────────────────────────────────────────────────────────
    manifest = write_manifest(layers, out_dir)
    write_verilog_pkg(layers, manifest, out_dir)

    # ── Verification ──────────────────────────────────────────────────────────
    if args.verify and float_model is not None:
        verify_extraction(float_model, layers)
    elif args.verify and float_model is None:
        print("[Verify] Skipped: float model not available in PTQ path")

    # ── Summary ───────────────────────────────────────────────────────────────
    print_summary(layers, out_dir)


if __name__ == "__main__":
    main()

In [ ]:
!python weight_extractor.py --verify